In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm 
from scipy.stats import f
import warnings
warnings.filterwarnings("ignore")

In [2]:
# 检验时间选择2011-2021年
# 读取三因子数据   需要 日期、市场溢酬因子_总市值加权、,市值因子_总市值加权、账面市值比_总市值加权
# 这里读取到的就是FFF的月度数据
data_factors = pd.read_excel('./data/que1/FFF_week_data/RESSET_THRFACDAT_MONTHLY_1.xls',usecols=[2,6,7,8])
data_factors.columns = ['date', 'mkt', 'smb', 'hml']
data_factors['date'] = pd.to_datetime(data_factors['date'])
data_factors['yearmonth'] = data_factors['date'].dt.strftime('%Y%m')

# 提取出2011年1月到2021年12月的FFF数据
# 设置时间范围的起始和结束
start_yearmonth = '201101'
end_yearmonth = '202112'
# 筛选出符合时间范围的数据
data_factors_filtered = data_factors[(data_factors['yearmonth'] >= start_yearmonth) & (data_factors['yearmonth'] <= end_yearmonth)]
data_factors_filtered

,date,mkt,smb,hml,yearmonth
114,2011-01-31,-0.0109,-0.0062,0.0684,201101
115,2011-02-28,0.0370,0.0595,-0.0394,201102
116,2011-03-31,0.0047,0.0103,0.0241,201103
117,2011-04-29,-0.0087,-0.0089,0.0155,201104
118,2011-05-31,-0.0612,-0.0113,-0.0032,201105
...,...,...,...,...,...
241,2021-08-31,0.0488,0.0068,0.0840,202108
242,2021-09-30,0.0094,-0.0205,0.0468,202109
243,2021-10-29,-0.0096,-0.0056,-0.0988,202110
244,2021-11-30,-0.0025,0.1028,-0.0584,202111


In [3]:
#month_data中是七个行业指数的日度数据，后面又月标识符，根据月标识符筛选出月度数据
HYindex_data_daily = pd.read_excel('./data/que1/month_data/month_data.xls',usecols=[1,2,3,4])
HYindex_data_daily.columns = ['idxname', 'date', 'endPrice','month_biaoji']
HYindex_data_daily['date'] = pd.to_datetime(HYindex_data_daily['date'])
HYindex_data_daily['yearmonth'] = HYindex_data_daily['date'].dt.strftime('%Y%m')
HYindex_data_daily.dropna(inplace=True)

# 这是选取的七个行业指数的名字
idxname = np.unique(HYindex_data_daily['idxname'].values)
idxname

array(['上证信息', '上证医药', '上证工业', '上证材料', '上证消费', '上证能源', '上证通信'],
      dtype=object)

In [4]:
# 从所有数据中分别提取出每个行业指数的数据并处理
def idx_dataProcess(idxname):
    idxData = HYindex_data_daily[HYindex_data_daily['idxname'] == idxname]
    idxData['return'] = np.log(idxData['endPrice']) - np.log(idxData['endPrice'].shift(periods=1))
    select_columns = ['idxname','date','yearmonth','return']
    idxData = idxData[select_columns]
    idxData.columns = ['idxname','date','yearmonth','return_'+idxname]
    # 筛选出符合时间范围的数据
    start_yearmonth = '201101'
    end_yearmonth = '202112'
    idxData_filtered = idxData[(idxData['yearmonth'] >= start_yearmonth) & (idxData['yearmonth'] <= end_yearmonth)]
    return idxData_filtered

In [5]:
data0_xinxi = idx_dataProcess(idxname=idxname[0])
data1_yiyao = idx_dataProcess(idxname=idxname[1])
data2_gongye = idx_dataProcess(idxname=idxname[2])
data3_cailiao = idx_dataProcess(idxname=idxname[3])
data4_xiaofei = idx_dataProcess(idxname=idxname[4])
data5_nengyuan = idx_dataProcess(idxname=idxname[5])
data6_tongxin = idx_dataProcess(idxname=idxname[6])

In [6]:
# 读取月无风险收益数据
data_rf = pd.read_excel('./data/que1/RiskFreeReturn/2-rf(1).xls',usecols=[1,2])
data_rf.columns = ['date', 'rfreturn']
data_rf['date'] = pd.to_datetime(data_rf['date'])
data_rf['yearmonth'] = data_rf['date'].dt.strftime('%Y%m')

In [7]:
# 将所有数据拼接
data_matrix = pd.merge(left=data_factors[['yearmonth', 'date', 'mkt', 'smb', 'hml']],
                      right=data0_xinxi[['yearmonth', 'return_上证信息']],
                      on=['yearmonth'],
                      how='inner')

data_matrix = pd.merge(left=data_matrix, 
                       right=data1_yiyao[['yearmonth', 'return_上证医药']],
                       on=['yearmonth'],
                       how='inner')

data_matrix = pd.merge(left=data_matrix, 
                       right=data2_gongye[['yearmonth', 'return_上证工业']],
                       on=['yearmonth'],
                       how='inner')

data_matrix = pd.merge(left=data_matrix, 
                       right=data3_cailiao[['yearmonth', 'return_上证材料']],
                       on=['yearmonth'],
                       how='inner')

data_matrix = pd.merge(left=data_matrix, 
                       right=data4_xiaofei[['yearmonth', 'return_上证消费']],
                       on=['yearmonth'],
                       how='inner')

data_matrix = pd.merge(left=data_matrix, 
                       right=data5_nengyuan[['yearmonth', 'return_上证能源']],
                       on=['yearmonth'],
                       how='inner')

data_matrix = pd.merge(left=data_matrix, 
                       right=data6_tongxin[['yearmonth', 'return_上证通信']],
                       on=['yearmonth'],
                       how='inner')

data_matrix = pd.merge(left=data_matrix, 
                       right=data_rf[['yearmonth', 'rfreturn']],
                       on=['yearmonth'],
                       how='inner')
data_matrix.columns = ['yearmonth', 'date', 'mkt', 'smb', 'hml','xinxi','yiyao','gongye','cailiao','xiaofei','nengyuan','tongxin','rfreturn']
data_matrix.dropna(inplace=True)
data_matrix.sort_values(by='date', inplace=True)
data_matrix

,yearmonth,date,mkt,smb,hml,xinxi,yiyao,gongye,cailiao,xiaofei,nengyuan,tongxin,rfreturn
0,201101,2011-01-31,-0.0109,-0.0062,0.0684,-0.086881,-0.089999,0.065207,-0.061090,-0.084314,-0.043189,-0.101527,0.003930
1,201102,2011-02-28,0.0370,0.0595,-0.0394,0.105793,0.040807,0.023064,0.120869,0.074379,0.047244,0.115611,0.004320
2,201103,2011-03-31,0.0047,0.0103,0.0241,-0.065658,-0.062895,-0.056968,0.035423,-0.031715,0.029437,-0.078072,0.003581
3,201104,2011-04-29,-0.0087,-0.0089,0.0155,-0.074902,-0.005231,-0.017266,-0.034126,-0.030729,-0.009774,-0.064598,0.003592
4,201105,2011-05-31,-0.0612,-0.0113,-0.0032,-0.078339,-0.051661,-0.073008,-0.073559,-0.024820,-0.071305,-0.074401,0.003795
...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,202108,2021-08-31,0.0488,0.0068,0.0840,-0.153716,-0.118951,0.134042,0.106857,-0.048163,0.192425,0.020611,0.001969
128,202109,2021-09-30,0.0094,-0.0205,0.0468,-0.023142,0.057346,-0.036536,-0.125211,0.098271,0.095373,-0.048512,0.001980
129,202110,2021-10-29,-0.0096,-0.0056,-0.0988,0.027997,-0.032733,0.033296,-0.028172,0.058473,-0.149550,0.056847,0.002027
130,202111,2021-11-30,-0.0025,0.1028,-0.0584,0.049188,0.032915,-0.055749,0.004111,-0.012571,-0.072401,0.114360,0.002055


In [8]:
# 提取出自变量（三因子的值）
x = data_matrix.loc[:, ['mkt', 'smb', 'hml']].values
ret_rf = data_matrix.loc[:, ['rfreturn']].values  # 无风险收益率
ret_pf = data_matrix.loc[:, ['xinxi']].values  # 上证信息的收益率

# 单资产检验
X = sm.add_constant(x)  # 给自变量加上截距项
Y = ret_pf - ret_rf  # 上证信息的超额收益率作为因变量
model = sm.OLS(Y, X)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.777
Model:                            OLS   Adj. R-squared:                  0.771
Method:                 Least Squares   F-statistic:                     148.3
Date:                Sun, 16 Jun 2024   Prob (F-statistic):           1.78e-41
Time:                        13:45:01   Log-Likelihood:                 225.07
No. Observations:                 132   AIC:                            -442.1
Df Residuals:                     128   BIC:                            -430.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0038      0.004     -0.962      0.3

其他指数的单资产检验同上

In [9]:
# 多资产检验
T = len(Y)  # 时间序列长度
N = 10  #资产数量
K = 3 # 因子数量
# 所有指数收益率都减去无风险收益率
ym = data_matrix.iloc[:, 5:12].values - data_matrix.loc[:, ['rfreturn']].values
xm = sm.add_constant(x)
xmTxm = np.dot(np.transpose(xm), xm)
xmTym = np.dot(np.transpose(xm), ym)
# AB_hat 是通过最小二乘法估计得到的参数矩阵，包含了每个因子的系数以及截距项
AB_hat = np.dot(np.linalg.inv(xmTxm), xmTym)
ALPHA = AB_hat[0]
print(ALPHA)

[-0.00375772  0.0008876  -0.00392978 -0.00560766  0.00540592 -0.01218921
 -0.00610399]


In [10]:
# RESD 是残差，即实际超额收益率与模型预测超额收益率的差
RESD = ym - np.dot(xm, AB_hat)
# COV 是残差的协方差矩阵
COV = np.dot(np.transpose(RESD), RESD)/T
# invCOV 是协方差矩阵的逆矩阵
invCOV = np.linalg.inv(COV)

# fs 是除去常数项后的因子数据，常数项是第0列
fs = xm[:, [1, 2, 3]]
# 因子的样本均值
muhat = np.mean(fs, axis=0).reshape((3, 1))
fs = fs - np.mean(fs, axis=0)
omegahat = np.dot(np.transpose(fs), fs)/T
invOMG = np.linalg.inv(omegahat)

xxx = np.dot(np.dot(np.transpose(muhat), invOMG), muhat)
print(xxx)
yyy = np.dot(np.dot(ALPHA, invCOV), np.transpose(ALPHA))
print(yyy)
# GRS检验
GRS = (T-N-K)/N/(1+xxx[0][0])*yyy
print(GRS)
pvalue = 1 - f.cdf(GRS, N, T-N-K)
print(pvalue)

[[0.00965435]]
0.18792337147052698
2.214904655196622
0.021233695230779892
